# Hướng Dẫn Giải Thích Chi Tiết: `scripts/run_backtest.py`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của hệ thống mô phỏng kiểm thử giao dịch thực tế (Backtest Simulator). Đây là module quan trọng giúp đánh giá hiệu suất của mô hình AI khi đưa vào thực tế giao dịch.

---

## 🔍 1. Tổng Quan Về Hệ Thống Backtest Thực Tế

Trong quant research, việc chỉ tính toán độ chính xác (Accuracy, RMSE) là chưa đủ. Một mô hình có độ chính xác 60% vẫn có thể cháy tài khoản nếu không quản lý tốt phí giao dịch và độ trượt giá.

Hệ thống backtest này áp dụng các nguyên lý thực tế bao gồm:
1. **Phí Giao Dịch & Thuế (Commissions & Tax):** Khấu trừ tỷ lệ phần trăm trên mỗi lệnh mua và bán. Thị trường VN được cài đặt phí bán cao hơn do có 0.1% thuế TNCN.
2. **Độ Trượt Giá Khớp Lệnh (Slippage):** Khi đặt lệnh mua số lượng lớn, giá khớp thực tế sẽ bị đẩy lên cao hơn giá đóng cửa, và ngược lại khi bán. Hệ thống áp dụng tỷ lệ trượt giá để mô phỏng thực tế.
3. **Kiểm Định Cuốn Chiếu (Walk-Forward Validation):** Thay vì đánh giá gộp một giai đoạn, hệ thống chia tập Test thành 3 Window độc lập cuốn chiếu theo thời gian để kiểm tra độ ổn định của thuật toán qua các chu kỳ thị trường khác nhau.

In [1]:
import sys
import os
import numpy as np
import pandas as pd

# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath('..'))
print("Import thư viện thành công!")

Import thư viện thành công!


## ⚙️ 2. Quy Tắc Giao Dịch Qua Đêm (Overnight Trading Rules)

Mô hình giả lập một chiến lược **Overnight Trading**:
- **Mở vị thế (BUY):** Chiều hôm nay, khi phiên sắp đóng cửa (Close), mô hình sẽ đưa ra dự đoán xu hướng cho Open ngày mai. Nếu cả 2 mô hình (XGBoost Lai và Transformer) đồng thuận báo TĂNG (> +0.25%), hệ thống sẽ giải ngân 100% tài sản mua cổ phiếu tại giá Close (chịu phí giao dịch + trượt giá).
- **Đóng vị thế (SELL):** Sáng ngày hôm sau, hệ thống bán toàn bộ số cổ phiếu ngay khi mở cửa (Open) (chịu phí giao dịch + trượt giá) và giữ tiền mặt cho đến khi có tín hiệu mua tiếp theo.

In [2]:
# Giả lập tính toán chi phí giao dịch trên 1 lệnh mua
capital = 100000000.0  # Vốn ban đầu: 100 triệu VNĐ
close_price = 62000.0   # Giá đóng cửa hôm nay
commission_pct = 0.0020 # Phí giao dịch 0.20% (áp dụng cho thị trường VN)
slippage_pct = 0.0010   # Độ trượt giá mua 0.10%

# Giá mua thực tế sau trượt giá
buy_price = close_price * (1 + slippage_pct)
# Số cổ phiếu mua được sau khi trừ phí giao dịch
shares = (capital * (1 - commission_pct)) / buy_price

print(f"Vốn ban đầu: {capital:,.2f} VNĐ")
print(f"Giá Close: {close_price:,.2f} VNĐ -> Giá khớp mua thực tế: {buy_price:,.2f} VNĐ")
print(f"Số lượng cổ phiếu mua được: {shares:.2f} cổ phiếu")

Vốn ban đầu: 100,000,000.00 VNĐ
Giá Close: 62,000.00 VNĐ -> Giá khớp mua thực tế: 62,062.00 VNĐ
Số lượng cổ phiếu mua được: 1608.07 cổ phiếu


## 📊 3. Đánh Giá Hiệu Suất & Walk-Forward Validation

Hệ thống chia tập Test (693 phiên) thành 3 giai đoạn cuốn chiếu (Windows):
- **Sharpe Ratio (Annualized):** Đo lường tỷ suất lợi nhuận trên mỗi đơn vị rủi ro.
- **Maximum Drawdown (MDD):** Mức sụt giảm tài sản lớn nhất từ đỉnh gần nhất (đo lường độ rủi ro cháy tài khoản).

In [3]:
# Chạy trực tiếp module backtest cho cổ phiếu Vinamilk (VNM.VN)
import subprocess
import sys
import os

script_path = os.path.abspath(os.path.join('..', 'scripts', 'run_backtest.py'))
print("🚀 Đang chạy Backtest cho VNM.VN... (Có thể mất 1-2 phút do nạp mô hình Deep Learning)")
result = subprocess.run(
    [sys.executable, script_path, 'VNM.VN'],
    capture_output=True,
    text=True,
    encoding='utf-8'
)

print("=== KẾT QUẢ BACKTEST TỪ CONSOLE ===")
print(result.stdout)


🚀 Đang chạy Backtest cho VNM.VN... (Có thể mất 1-2 phút do nạp mô hình Deep Learning)


=== KẾT QUẢ BACKTEST TỪ CONSOLE ===

📊 BACKTEST: VNM.VN
Đang tải tỷ giá USD/VND (USDVND=X) từ 2012-01-01 đến 2026-05-20...
Đang tải Yahoo Finance: VNM.VN...
  Yahoo Finance: 3654 phiên
Đang nạp dữ liệu trường: C:\Users\ACER\Documents\Stock-Opening-Price-Prediction\data\raw\VNM_prices.csv
  Tổng: 3535 phiên (2012-03-20 → 2026-05-19)
Tính macro features...
  [NEWS] Tải và phân tích cảm xúc tin tức cho VNM.VN...
📥 Đang tải dữ liệu cổ tức cho VNM.VN từ yfinance...
   => Đã tải 36 đợt chia cổ tức cho VNM.VN
Đã lưu cache: C:\Users\ACER\Documents\Stock-Opening-Price-Prediction\data\VNM.VN_processed.csv
Sẵn sàng với 3531 phiên.

📊 Split 80/20 (Purge Gap: 45):
   🔹 Train: 2788 mẫu
   🔸 Test : 653 mẫu
   📁 Model trained : 2026-06-12 05:00
   📁 Tuning config : 2026-06-11 04:45
   ✅ Model đã train SAU tuning — OK
   [PREDICT] Đang chạy dự báo trên tập test...
   [DEBUG] Corr XGB-Actual (T+1)=+0.0335 | Corr Trans-Actual (T+1)=+0.0079
   [DEBUG] XGB mean (T+1)=0.00080 | Trans mean (T+1)=-0.00175 | A